## Setup

In [ ]:
import pandas as pd  # tables of data

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## 2026: NEET rates by age group and sex

In [ ]:
def wmean(data, var):  # weighted mean of one variable
    d = data.dropna(subset=[var])  # leave out people with a missing value
    return (d[var] * d["asecwt"]).sum() / d["asecwt"].sum()  # sum of value x weight, divided by sum of weights

if "agegrp" not in df.columns:  # make age groups only if the file does not have them
    df["agegrp"] = pd.cut(df["age"], bins=[16, 20, 25, 30], labels=["16-19", "20-24", "25-29"], right=False)  # 16-19, 20-24, 25-29

df["sex"] = pd.Categorical(df["female"].map({0: "Men", 1: "Women"}), categories=["Men", "Women"], ordered=True)  # words for female, Men first

df26 = df[df["year"] == 2026].copy()  # keep only the 2026 survey
known = df26.dropna(subset=["neet"])  # keep only people whose neet is known

rows = {}  # empty dictionary, one row per age group and sex
for (age, sex), group in known.groupby(["agegrp", "sex"], observed=True):  # loop over each age group and sex
    rows[f"{age} {sex}"] = {  # row label, e.g. "16-19 Men"
        "Unemployed NEET (%)": wmean(group, "unemployed_neet") * 100,  # weighted percent unemployed NEET
        "Out-of-labor-force NEET (%)": wmean(group, "nilf_neet") * 100,  # weighted percent out-of-labor-force NEET
        "Total NEET (%)": wmean(group, "neet") * 100,  # weighted percent NEET
    }  # end of one row

table = pd.DataFrame(rows).T  # dictionary -> table, one row per age group and sex
people = known.groupby(["agegrp", "sex"], observed=True).size()  # unweighted respondents in each row
people.index = [f"{age} {sex}" for age, sex in people.index]  # same row labels as the table

gap = (table["Unemployed NEET (%)"] + table["Out-of-labor-force NEET (%)"] - table["Total NEET (%)"]).abs()  # difference in each row, before rounding
print("Table. NEET rates in 2026, % of people (weighted with asecwt). Rows: age group and sex.")  # table title
print(table.to_string(float_format="{:.1f}".format))  # table, 1 decimal
print()  # empty line
print("Check: unemployed NEET + out-of-labor-force NEET = total NEET in every row:", bool((gap < 1e-9).all()))  # True or False
print("Largest difference in any row (before rounding):", gap.max())  # should be about 0
print()  # empty line
print("Unweighted number of respondents in each row (neet not missing).")  # counts title
print(people.to_string())  # counts, whole numbers

**What one cell means.** Take the row "20-24 Women" and the column "Unemployed NEET (%)". Among women aged 20-24 in the 2026 survey whose NEET status is known, the cell is the percent who were NEET and unemployed: not employed, looking for work, and not in school. The column "Out-of-labor-force NEET (%)" is the percent who were NEET and not looking for work. The column "Total NEET (%)" is the percent who were NEET, which is the first two columns added together. People are counted by how many people in the country they represent (`asecwt`), so each number describes the population, not just the people surveyed. A cell of 10.0 means 10 out of every 100 such people. The counts under the table show how many survey respondents sit behind each row.